# IMC Prosperity 4 — Round 3 EDA

**Underlying:** `VELVETFRUIT_EXTRACT`
**Options:** `VEV_4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500` (calls on the underlying)
**Side product:** `HYDROGEL_PACK`
**Currency:** `XIRECS`

This notebook walks through five EDA passes:

1. **Data integrity** — schema, shapes, missingness
2. **Underlying behaviour** — drift, vol, autocorrelation
3. **Option chain shape** — calls vs puts, intrinsic, dust strikes
4. **Implied volatility surface** — IV per strike, smile, TTE convention
5. **HYDROGEL_PACK + parity / no-arb checks**

Run cells top-to-bottom the first time. After that you can re-run any section independently because each pass loads from the pickled outputs of the data-load step.

> **Note on conventions.** Throughout the notebook I work in *tick time* rather than calendar time when possible. The IMC TTE convention is opaque, but the only thing that actually matters for option pricing is the product `σ × √T`. Working in total-vol space removes the TTE guesswork.


## 0 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
from pathlib import Path

# --- EDIT THIS to point to wherever your CSVs are ---
DATA_DIR = Path("./data")  # e.g. Path("/Users/mark/Downloads/round3")
# CSVs expected: prices_round_3_day_{0,1,2}.csv  trades_round_3_day_{0,1,2}.csv

OUT_DIR = Path("./out")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## Pass 1 — Data integrity

We are loading 3 days of prices (around 120k rows each) and 3 days of trades (around 1.3k rows each), separated by `;`. Each prices row is one product at one timestamp, with up to 3 levels of order book depth on each side.

The first thing I always want to know:
- Are all products present every day?
- Is the timestamp grid uniform?
- Are there any unexpected NaNs?
- Are there any duplicate (timestamp, product) pairs?

If any of these are off, every later pass becomes suspect.

In [ ]:
# --- Load all 6 CSVs ---
prices_files = [DATA_DIR / f"prices_round_3_day_{d}.csv" for d in range(3)]
trades_files = [DATA_DIR / f"trades_round_3_day_{d}.csv" for d in range(3)]

prices = pd.concat([pd.read_csv(f, sep=";") for f in prices_files], ignore_index=True)

# trades files don't have a `day` column — add one
tlist = []
for d, f in enumerate(trades_files):
    t = pd.read_csv(f, sep=";")
    t["day"] = d
    tlist.append(t)
trades = pd.concat(tlist, ignore_index=True)

# Build a "global timestamp" so we can plot all 3 days on one axis
prices["gts"] = prices["day"] * 1_000_000 + prices["timestamp"]
trades["gts"] = trades["day"] * 1_000_000 + trades["timestamp"]

print(f"prices: {prices.shape}")
print(f"trades: {trades.shape}")
prices.head(3)

In [ ]:
# --- Schema sanity ---
print("Products x day (rows):")
print(prices.groupby(["day", "product"]).size().unstack(fill_value=0))
print()
print("Timestamps per day:")
print(prices.groupby("day")["timestamp"].nunique().to_dict())
print()
print("Timestamp step (should be 100):", np.diff(sorted(prices.query("day==0")["timestamp"].unique()))[:3])

In [ ]:
# --- Missingness ---
# Levels 2 and 3 of the book are often absent; level 1 should always be there.
prices.isna().sum()

In [ ]:
# --- Duplicate (gts, product) check ---
dups = prices.duplicated(subset=["gts", "product"]).sum()
print(f"Duplicate (timestamp, product) rows: {dups}")
print(f"Trades with named buyer/seller: buyer={trades['buyer'].notna().sum()}, seller={trades['seller'].notna().sum()}")

In [ ]:
# Cache everything as pickle so subsequent passes can be re-run independently.
prices.to_pickle(OUT_DIR / "prices.pkl")
trades.to_pickle(OUT_DIR / "trades.pkl")
print("cached.")

**What to look for in the output above:**
- `Products x day` should be a 3x12 table all reading 10000 — every product, every day, full 10k timestamps.
- Levels 2 and 3 will have many NaNs (book depth is sparse). Level 1 columns should have 0 NaNs.
- `mid_price` should always be present.
- Duplicates should be 0.
- `buyer` / `seller` will both be 0 — the bots are anonymised in the capsule data. We can see *that* a trade happened, not *who* did it.

If any of those fail, stop and investigate before going further.

---

## Pass 2 — The underlying: VELVETFRUIT_EXTRACT

Before we touch a single option we need to know what the underlying *does*. Specifically:
- Is it stationary or drifting?
- What is its per-tick volatility?
- Is the spread tight or wide?
- Is there short-term mean reversion (real, or just bid-ask bounce)?
- Does the vol regime shift across the 3 days?

Whatever pricing model we end up using on the options assumes some answer to these questions. If we model the underlying as Brownian motion but it is actually trending, our IVs will be biased. So this pass exists to validate the modelling assumption.

In [ ]:
prices = pd.read_pickle(OUT_DIR / "prices.pkl")

vf = (prices.query("product == 'VELVETFRUIT_EXTRACT'")
            .sort_values("gts")
            .reset_index(drop=True)
            .copy())
vf["spread"] = vf["ask_price_1"] - vf["bid_price_1"]
vf["ret"]    = vf["mid_price"].diff()
vf["logret"] = np.log(vf["mid_price"]).diff()

print("Per-day mid stats:")
print(vf.groupby("day")["mid_price"].agg(["first", "last", "mean", "std", "min", "max"]).round(2))
print()
print("Spread stats:")
print(vf["spread"].describe().round(3))
print()
print(f"Per-tick std of mid changes: {vf['ret'].std():.4f}")
print(f"Per-tick std of log-returns: {vf['logret'].std():.6e}")
print()
ac = [vf["ret"].autocorr(lag=k) for k in [1, 2, 5, 10, 20, 50]]
print(f"Autocorr(returns) at lags 1, 2, 5, 10, 20, 50: {[round(x, 4) for x in ac]}")

In [ ]:
# Persist for later passes
vf.to_pickle(OUT_DIR / "vf.pkl")

In [ ]:
# --- Plot 1: full mid path
fig, ax = plt.subplots(figsize=(12, 4))
for d in range(3):
    sub = vf.query("day == @d")
    ax.plot(sub["gts"], sub["mid_price"], lw=0.5, label=f"day {d}")
ax.set_title("VELVETFRUIT_EXTRACT mid_price (all 3 days)")
ax.set_xlabel("global timestamp")
ax.set_ylabel("mid")
ax.legend()
plt.tight_layout()

In [ ]:
# --- Plot 2: distribution of per-tick mid changes (log y so tails are visible)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(vf["ret"].dropna(), bins=80)
ax.set_yscale("log")
ax.set_title(f"Per-tick mid change distribution (std={vf['ret'].std():.2f})")
plt.tight_layout()

In [ ]:
# --- Plot 3: rolling realised vol
window = 500
vf["rv500"] = vf["logret"].rolling(window).std()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vf["gts"], vf["rv500"], lw=0.6)
ax.set_title(f"Rolling {window}-tick realised vol of log-returns")
ax.set_xlabel("global timestamp")
plt.tight_layout()

**What you should be reading off these:**

- **Path plot.** It wanders 5,200 -> 5,300, no clear trend across days, day 2 ends a bit higher than start. That is a random-walk-ish profile, not a stationary asset like Emeralds. **No "buy below 5,250, sell above 5,250" play** — you would be picking up nickels in front of a steamroller because there is no anchor.

- **Histogram.** Discrete bins because mid moves in 0.5 increments (half-tick). Tails are roughly Gaussian — log scale just makes outliers visible.

- **Rolling vol.** Stays in a narrow band (around 0.0002 log-return per tick). **No vol-regime shifts.** That is actually convenient — a single sigma estimate is reasonable for option pricing across the whole sample.

- **Autocorrelation.** Lag-1 is around -0.16 — that is bid-ask bounce, not tradeable mean reversion. Lag-2 onwards is essentially zero. Do not waste time on a moving-average crossover here.

**Conclusion:** the underlying is well-approximated as random walk with constant vol. **Black–Scholes is a defensible pricing baseline.**

> **TRY THIS:** swap `window=500` for `100` and `2000` to see how vol estimates change with the smoothing window. Smaller windows are noisier but pick up regime shifts faster.

---

## Pass 3 — Option chain shape

Two structural questions to answer before doing anything quantitative with the options:

1. **Are these calls or puts?** The naming `VEV_<strike>` does not tell us. We will figure it out by comparing option mids against call-intrinsic and put-intrinsic curves at a single timestamp.
2. **Which strikes are tradeable?** Some strikes will have meaningful time value (where the action is) and some will be either pure-intrinsic (deep ITM, behaves like delta-1 underlying) or near-zero dust (deep OTM, no signal).

In [ ]:
prices = pd.read_pickle(OUT_DIR / "prices.pkl")
vf     = pd.read_pickle(OUT_DIR / "vf.pkl")

vev_products = sorted(
    [x for x in prices["product"].unique() if x.startswith("VEV_")],
    key=lambda s: int(s.split("_")[1])
)
strikes_all = [int(p.split("_")[1]) for p in vev_products]
print("VEV strikes:", strikes_all)

In [ ]:
# Per-product summary across the entire 3-day sample
rows = []
for p in vev_products:
    sub = prices.query("product == @p")
    sp = sub["ask_price_1"] - sub["bid_price_1"]
    rows.append(dict(
        product       = p,
        strike        = int(p.split("_")[1]),
        mid_mean      = sub["mid_price"].mean(),
        mid_min       = sub["mid_price"].min(),
        mid_max       = sub["mid_price"].max(),
        mid_std       = sub["mid_price"].std(),
        spread_mean   = sp.mean(),
        bid_vol_med   = sub["bid_volume_1"].median(),
        ask_vol_med   = sub["ask_volume_1"].median(),
    ))
chain = pd.DataFrame(rows)
chain.round(2)

In [ ]:
# --- Decisive plot: mid vs strike at one snapshot, with intrinsic curves
# Take a midpoint snapshot
snap_gts = vf["gts"].iloc[len(vf) // 2]
snap = prices.query("gts == @snap_gts")
S_snap = float(snap.query("product == 'VELVETFRUIT_EXTRACT'")["mid_price"].iloc[0])

opts_snap = (snap[snap["product"].str.startswith("VEV_")]
             .assign(strike=lambda d: d["product"].str.split("_").str[1].astype(int))
             .sort_values("strike"))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(opts_snap["strike"], opts_snap["mid_price"], "o-", label=f"observed mid (S={S_snap:.1f})")
ax.plot(opts_snap["strike"], np.maximum(S_snap - opts_snap["strike"], 0), "--", label="call intrinsic = max(S-K, 0)")
ax.plot(opts_snap["strike"], np.maximum(opts_snap["strike"] - S_snap, 0), "--", label="put intrinsic = max(K-S, 0)")
ax.axvline(S_snap, color="red", ls=":", alpha=0.5, label="spot")
ax.set_xlabel("strike")
ax.set_ylabel("mid price")
ax.set_title("Option mid vs strike — decisive call-vs-put test")
ax.legend()
plt.tight_layout()

**Read this plot carefully — it is the single most important structural finding of the round.**

The observed mids hug the **call intrinsic line** (orange dashed) on the ITM side and decay toward zero past spot. This is unambiguously a **call payoff shape**. If they were puts, they would hug the green line instead.

So: **`VEV_K` is a European call on `VELVETFRUIT_EXTRACT` with strike `K`.**

In [ ]:
# --- Plot 2: option mid time series, log scale (so all strikes share an axis)
fig, ax = plt.subplots(figsize=(12, 5))
cmap = plt.cm.viridis
for i, p in enumerate(vev_products):
    sub = prices.query("product == @p").sort_values("gts")
    ax.plot(sub["gts"], sub["mid_price"], lw=0.4,
            color=cmap(i / (len(vev_products) - 1)), label=f"K={p.split('_')[1]}")
ax.set_yscale("log")
ax.set_title("VEV option mids over time (log y)")
ax.set_xlabel("global timestamp")
ax.set_ylabel("mid (log scale)")
ax.legend(ncol=2, fontsize=8, loc="center right")
plt.tight_layout()

**Triage of the chain (informed by the table and these plots):**

| Strike(s)    | Behaviour                                                  | Trading relevance                                       |
|--------------|------------------------------------------------------------|---------------------------------------------------------|
| 4000, 4500   | Mid ~ S - K (pure intrinsic, near-zero time value)         | Use as **synthetic underlying** for hedging delta       |
| 5000–5500    | Meaningful time value, varies through time                 | **Where the vol-trading game lives**                    |
| 6000, 6500   | Pinned at mid=0.5 (bid 0, ask 1), `mid_std` literally 0    | **Dust. Avoid.** No information, no liquidity           |

Notice that the bid/ask sizes on the dust strikes are still around 22 lots — there is a market-maker bot quoting them, but at 1-tick spreads with no movement, there is no edge to extract.

**Quick no-arb check:** at the snapshot, mids are strictly decreasing in K (1243.5 -> 743 -> 248 -> 160 -> 91.5 -> 45 -> 15 -> 5.5 -> 0.5 -> 0.5). Calls should be monotonically decreasing in strike — that holds. We would also want convexity in K, which I will check more rigorously in Pass 5.

> **TRY THIS:** pick three other timestamps (start, end, a random middle) and re-do the snapshot plot. The shape should be the same with shifted spot. If you ever see mid > intrinsic + something or mid violating monotonicity, that is a no-arb violation = an actual free-money trade.

---

## Pass 4 — Implied volatility surface

This is the meaty pass. The plan:

1. Define a vectorised Black–Scholes call and a vectorised Newton-iteration implied-vol solver.
2. Compute IV per strike, per timestamp.
3. **Crucial trick:** work in `sigma * sqrt(T)` (total vol) directly so we do not need to know the IMC TTE convention. The TTE assumption is a multiplicative constant on sigma across all strikes — it does not affect the *shape* of the smile.
4. Plot the smile, look for persistent cross-strike anomalies.

If you want to compare to "annualised IV" later, multiply `sigma*sqrt(T)` by `1/sqrt(TTE_in_years)` for whichever convention you settle on.

In [ ]:
prices = pd.read_pickle(OUT_DIR / "prices.pkl")
vf     = pd.read_pickle(OUT_DIR / "vf.pkl")

# Pivot to (gts) x (product) for fast vectorised work
wide_mid = prices.pivot_table(index="gts", columns="product", values="mid_price", aggfunc="first")
S = wide_mid["VELVETFRUIT_EXTRACT"].values.astype(float)
gts = wide_mid.index.values
print(f"wide_mid shape: {wide_mid.shape}")
print(f"S range: [{S.min()}, {S.max()}], mean {S.mean():.2f}")

In [ ]:
# --- BS call price and Newton implied-vol solver, both vectorised over time.
# Working in total-vol w = sigma * sqrt(T), so no TTE needed.
# In this parametrisation, BS becomes:
#   d1 = (ln(S/K) + 0.5 w^2) / w
#   d2 = d1 - w
#   C  = S * N(d1) - K * N(d2)

def bs_call_w(S, K, w):
    """BS call with w = sigma * sqrt(T). Vectorised over S/w."""
    out = np.maximum(S - K, 0.0)
    mask = w > 1e-12
    if np.any(mask):
        d1 = (np.log(S[mask] / K) + 0.5 * w[mask]**2) / w[mask]
        d2 = d1 - w[mask]
        out[mask] = S[mask] * norm.cdf(d1) - K * norm.cdf(d2)
    return out

def implied_w_newton(C, S, K, w0=0.05, max_iter=60, tol=1e-7):
    """Solve for w = sigma * sqrt(T) given call price C. Vectorised. Returns NaN where undefined."""
    w = np.full_like(C, w0)
    intrinsic = np.maximum(S - K, 0.0)
    bad = (C <= intrinsic + 1e-8) | (C >= S - 1e-8)
    for _ in range(max_iter):
        good = ~bad
        if not good.any():
            break
        d1 = (np.log(S[good] / K) + 0.5 * w[good]**2) / w[good]
        price = S[good] * norm.cdf(d1) - K * norm.cdf(d1 - w[good])
        vega  = S[good] * norm.pdf(d1)         # this is dC/dw, not dC/dsigma — same form
        vega  = np.where(vega < 1e-12, 1e-12, vega)
        diff  = price - C[good]
        new_w = np.clip(w[good] - diff / vega, 1e-5, 2.0)
        if np.max(np.abs(diff)) < tol:
            w[good] = new_w
            break
        w[good] = new_w
    w[bad] = np.nan
    return w

In [ ]:
# Compute implied total-vol for the strikes that actually have non-trivial time value
strikes_active = [5000, 5100, 5200, 5300, 5400, 5500]
w_grid = {}
for K in strikes_active:
    C = wide_mid[f"VEV_{K}"].values.astype(float)
    w_grid[K] = implied_w_newton(C, S, float(K))
    valid = (~np.isnan(w_grid[K])).sum()
    print(f"K={K}: mean sig*sqrt(T) = {np.nanmean(w_grid[K]):.5f}, std = {np.nanstd(w_grid[K]):.5f}, valid {valid}/{len(C)}")

w_df = pd.DataFrame(w_grid, index=gts)
w_df.to_pickle(OUT_DIR / "w_df.pkl")

In [ ]:
# --- Plot 1: total-vol time series per strike
fig, ax = plt.subplots(figsize=(12, 5))
for K in strikes_active:
    ax.plot(w_df.index, w_df[K], lw=0.3, alpha=0.7, label=f"K={K}")
ax.set_title("Implied total-vol sigma*sqrt(T) over time, per strike (TTE-free)")
ax.set_xlabel("global timestamp")
ax.set_ylabel("sigma*sqrt(T)")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()

**The key observation:** sigma*sqrt(T) monotonically *decreases* over time. That is exactly what time decay should look like — TTE is shrinking, so total vol shrinks even if per-tick sigma is constant. This is a sanity check that our pricing model is internally consistent.

If sigma*sqrt(T) were *flat* over time, that would mean the bots are not letting theta pull the prices down — i.e. they are keeping IV constant and letting theta accumulate as edge for whoever buys. Different game. Here we see decay, so options behave like options.

In [ ]:
# --- Plot 2: average smile (mean sigma*sqrt(T) vs strike)
mean_w = [w_df[K].mean() for K in strikes_active]
std_w  = [w_df[K].std()  for K in strikes_active]

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(strikes_active, mean_w, yerr=std_w, fmt="o-", capsize=3)
ax.axvline(S.mean(), color="red", ls=":", label=f"mean S = {S.mean():.0f}")
ax.set_xlabel("strike")
ax.set_ylabel("mean sigma*sqrt(T)")
ax.set_title("Average smile (mean +/- 1 std)")
ax.legend()
plt.tight_layout()

In [ ]:
# --- Plot 3: per-day smile shape
fig, ax = plt.subplots(figsize=(10, 5))
n = len(w_df)
for lo, hi, label in [(0, n//3, "day 0"), (n//3, 2*n//3, "day 1"), (2*n//3, n, "day 2")]:
    means = [w_df[K].iloc[lo:hi].mean() for K in strikes_active]
    ax.plot(strikes_active, means, "o-", label=label)
ax.set_title("Smile shape per day — does it persist?")
ax.set_xlabel("strike")
ax.set_ylabel("mean sigma*sqrt(T)")
ax.legend()
plt.tight_layout()

In [ ]:
# --- Plot 4: residuals from a per-timestamp quadratic fit in K
# Logic: at every timestamp, fit sigma*sqrt(T) as a quadratic in (K - mean_K). Compute the residual for each strike.
# Average residuals across time. A persistently negative residual = persistently cheap; positive = persistently rich.
K_arr = np.array(strikes_active, dtype=float)
X = np.column_stack([np.ones_like(K_arr), K_arr - K_arr.mean(), (K_arr - K_arr.mean())**2])

w_arr = np.array([w_df[K].values for K in strikes_active])  # (n_strikes, n_times)
resid_sum = np.zeros(len(strikes_active))
counts = 0
for t in range(w_arr.shape[1]):
    y = w_arr[:, t]
    if np.any(np.isnan(y)):
        continue
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid_sum += y - X @ beta
    counts += 1
mean_residual = resid_sum / counts

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(strikes_active, mean_residual, width=60)
ax.axhline(0, color="black", lw=0.5)
ax.set_title("Persistent vol mispricing per strike\n(negative = persistently cheap vs neighbours)")
ax.set_xlabel("strike")
ax.set_ylabel("mean residual in sigma*sqrt(T)")
plt.tight_layout()
for K, r in zip(strikes_active, mean_residual):
    print(f"  K={K}: mean residual = {r:+.6f}")

**The headline finding of this whole notebook lives in that residuals plot.**

After fitting a smooth quadratic smile at every single timestamp and averaging the residuals across all 30,000 timestamps, **K=5400 is persistently cheap by ~0.0015 sigma*sqrt(T) units** — which is roughly **5% in vol terms** vs its neighbours. Every other strike has |residual| <= 0.0007. K=5400 is more than 2x the next-largest deviation, **and consistently negative across all three days**.

This is the cleanest tradeable structural feature in the chain.

**Trade hypothesis (needs validation in pass 5 and via backtesting):**
A long-K=5400-vol vs short-K=5300/5500-vol play. Mechanically this is a butterfly: **buy 1 VEV_5400, sell ~0.5 VEV_5300, sell ~0.5 VEV_5500**. The position is long vega on 5400 and short vega on the wings. Net delta should be roughly hedged because 5300 and 5500 straddle 5400.

Caveats before getting excited:
- Need to check whether the "cheapness" actually shows up in the *bid* (not just the mid). If you can only buy at the ask, the edge might evaporate.
- Need to check fill probability / quote sizes.
- Need to check what happens when spot moves significantly — the anomaly might be regime-dependent on S.

> **TRY THIS:** look at the K=5400 anomaly conditioned on spot. Does the residual change sign when spot is near 5400 vs far from it? Bin the data by `(S - 5400)` and check.

> **TRY THIS:** repeat the fit with a cubic instead of quadratic — does the K=5400 anomaly survive? If it does, it is not just "the quadratic cannot fit a kink".

---

## Pass 5 — HYDROGEL_PACK and parity / no-arb checks

Two threads to close out:

1. **What is HYDROGEL_PACK?** It has not appeared in any of our pricing logic. Is it correlated with VELVETFRUIT_EXTRACT? Is it a stationary asset? Does it hedge anything?
2. **Static no-arb checks on the call chain.** At every timestamp, calls should satisfy:
   - **Monotonicity:** C(K1) >= C(K2) for K1 < K2
   - **Convexity:** C is convex in K (i.e. butterfly is non-negative)
   - **Bounds:** max(S-K, 0) <= C <= S

Any violation = literal free money on the order book. Worth knowing if they exist.

In [ ]:
prices = pd.read_pickle(OUT_DIR / "prices.pkl")
vf     = pd.read_pickle(OUT_DIR / "vf.pkl")
trades = pd.read_pickle(OUT_DIR / "trades.pkl")

hp = (prices.query("product == 'HYDROGEL_PACK'")
            .sort_values("gts")
            .reset_index(drop=True)
            .copy())
hp["spread"] = hp["ask_price_1"] - hp["bid_price_1"]

print("HYDROGEL_PACK summary:")
print(hp.groupby("day")["mid_price"].agg(["first", "last", "mean", "std", "min", "max"]).round(2))
print()
print("Spread:", hp["spread"].describe().round(2).to_dict())
print(f"Trades: {(trades['symbol']=='HYDROGEL_PACK').sum()}")

In [ ]:
# --- Plot HP and VF together to see if they move together
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(hp["gts"], hp["mid_price"], lw=0.5, label="HYDROGEL_PACK")
axes[0].set_ylabel("HYDROGEL_PACK mid")
axes[0].legend()
axes[1].plot(vf["gts"], vf["mid_price"], lw=0.5, color="orange", label="VELVETFRUIT_EXTRACT")
axes[1].set_ylabel("VEX mid")
axes[1].set_xlabel("global timestamp")
axes[1].legend()
plt.tight_layout()

In [ ]:
# --- Correlation: align on gts, look at level and return correlations
both = hp[["gts", "mid_price"]].rename(columns={"mid_price": "hp"}).merge(
    vf[["gts", "mid_price"]].rename(columns={"mid_price": "vf"}),
    on="gts"
)
print(f"Level correlation: {both[['hp','vf']].corr().iloc[0,1]:.4f}")
both["hp_ret"] = both["hp"].diff()
both["vf_ret"] = both["vf"].diff()
print(f"Return correlation: {both[['hp_ret','vf_ret']].corr().iloc[0,1]:.4f}")

**Interpreting HYDROGEL_PACK:**

- If level correlation is high *and* return correlation is high -> HP is essentially a tracker of VEX. You could use it as a hedge or a synthetic.
- If level correlation is high but return correlation is ~0 -> coincidence at the level (both happen to live in similar range), no real co-movement. Do not pair-trade.
- If both correlations are low -> HP is independent and probably its own stationary or trending asset that needs its own analysis.

**My prior:** based on past IMC rounds, "side products" introduced alongside an options round are often either (a) hedging instruments that correlate with the underlying, or (b) independent products from earlier rounds carried forward. Look at the path plot. If HP looks stationary around a fixed level (like Emeralds), it is its own trader. If it tracks VEX, it is a hedge candidate.

> **TRY THIS:** if HP looks stationary, characterise it the same way you did VEX in Pass 2 — fit per-tick std, autocorrelation, spread. A simple market-maker on a stationary asset is usually one of the easier sources of PnL in IMC.

In [ ]:
# --- Static no-arb checks on the call chain at each timestamp
wide_mid = prices.pivot_table(index="gts", columns="product", values="mid_price", aggfunc="first")
S = wide_mid["VELVETFRUIT_EXTRACT"].values.astype(float)

# Use all 10 strikes for the no-arb checks
strikes_full = sorted(int(p.split("_")[1]) for p in prices["product"].unique() if p.startswith("VEV_"))
C_arr = np.column_stack([wide_mid[f"VEV_{K}"].values.astype(float) for K in strikes_full])
K_arr = np.array(strikes_full, dtype=float)

# 1) Monotonicity: C[i] >= C[i+1] (calls decreasing in K)
mono_violations = (np.diff(C_arr, axis=1) > 0).sum(axis=0)
# 2) Convexity (butterfly): C[i-1] - 2*C[i] + C[i+1] >= 0 for interior strikes
butter = C_arr[:, :-2] - 2*C_arr[:, 1:-1] + C_arr[:, 2:]
butter_violations = (butter < -1e-8).sum(axis=0)
# 3) Lower bound: C >= max(S-K, 0)
lb = np.maximum(S[:, None] - K_arr[None, :], 0.0)
lb_violations = (C_arr < lb - 1e-8).sum(axis=0)
# 4) Upper bound: C <= S
ub_violations = (C_arr > S[:, None] + 1e-8).sum(axis=0)

print(f"Total timestamps: {len(S)}\n")
print("Monotonicity violations (C should be down in K) — count between adjacent strikes:")
for i, K in enumerate(strikes_full[:-1]):
    print(f"  C(K={K}) > C(K={strikes_full[i+1]}): {mono_violations[i]}")
print()
print("Butterfly (convexity) violations — count for each interior strike:")
for i, K in enumerate(strikes_full[1:-1]):
    print(f"  K={K}: {butter_violations[i]}")
print()
print("Lower-bound (C < intrinsic) violations per strike:")
for i, K in enumerate(strikes_full):
    print(f"  K={K}: {lb_violations[i]}")
print()
print("Upper-bound (C > S) violations per strike:")
for i, K in enumerate(strikes_full):
    print(f"  K={K}: {ub_violations[i]}")

**Interpreting the no-arb checks:**

All four counts should be **zero** in a well-priced market. Any non-zero number means there is at least one timestamp where the *mid prices* admit an arbitrage. Two important nuances:

- **Mid violations are not the same as tradeable arbitrage.** Mids can violate when the bid-ask spread is wide enough that the actual bids and asks do not admit the trade. To turn this into a real trading signal you need to redo the check using `bid_price_1` and `ask_price_1` instead of mid: a butterfly arbitrage exists *for real* when `ask(K-1) + ask(K+1) - 2*bid(K) < 0` (you pay both wings asks, sell middle bid, and pocket negative cost).

- **Monotonicity violations near dust strikes are usually noise.** K=6000 and K=6500 are both pinned at mid=0.5; their adjacency check is meaningless. Focus on the active strikes 5000-5500.

If you see consistent butterfly violations on the active strikes, that is a legitimate vol-arb edge — it is *exactly* the K=5400 effect from Pass 4 showing up as a static-arb signature.

> **TRY THIS:** re-run the convexity check with `bid_price_1` and `ask_price_1`. The trade is: at any timestamp where `2*bid(5400) > ask(5300) + ask(5500)`, you sell 2 VEV_5400 (collect 2*bid), buy 1 VEV_5300 + 1 VEV_5500 (pay 2 asks), pocket the difference. Wait for convergence. Count how many timestamps offer that trade and at what size.

---

## Summary of findings

| Pass | Finding                                                                                            | Implication                                                       |
|------|----------------------------------------------------------------------------------------------------|-------------------------------------------------------------------|
| 1    | Clean dataset: 3 days x 10k timestamps x 12 products                                               | No data hygiene work needed                                       |
| 2    | VEX is random-walk-like, sigma stable across days, lag-1 autocorr is bid-ask bounce only           | Black–Scholes is defensible; do not trade VEX as mean-reversion   |
| 3    | VEV_K are calls. Strikes 4000-4500 are pure intrinsic, 5000-5500 active, 6000-6500 dust            | Vol-trading game lives on strikes 5000-5500                       |
| 4    | **K=5400 is persistently cheap in sigma*sqrt(T) by ~5% vs smooth smile** across all 3 days         | **Long-K=5400 / short-K=5300+5500 butterfly is the headline edge** |
| 5    | HP correlation with VEX (TBD locally), no-arb violations to check on bid/ask not mid               | Determine HP role; convert mid-arbs into bid/ask-arbs to trade    |

## Where to go next

- **Validate the K=5400 trade.** Build a lightweight backtester that: (a) detects when `ask(5300) + ask(5500) - 2*bid(5400)` is negative, (b) lifts the trade, (c) holds until the spread closes, (d) tracks PnL with spread costs and position limits.
- **Decide on HP role.** If correlated, it is a hedge. If independent and stationary, it is a market-making line.
- **Build a vol-pricing module for `Trader.py`** that consumes `S`, every `VEV_K` quote, and outputs:
  - per-strike fitted sigma*sqrt(T) (the smooth smile)
  - residual vs fit (the trade signal)
  - implied delta per strike (for hedging)
- **Do not forget execution.** Even a clean structural anomaly can lose money if execution is bad. Refer to your repo `execution/` notes — take/clear/make schema applies.

## Files written
- `out/prices.pkl`, `out/trades.pkl`, `out/vf.pkl` — cached frames
- `out/w_df.pkl` — per-strike, per-timestamp sigma*sqrt(T)
